# PhantomLM — Training Notebook
**Phone-native hybrid Mamba-Transformer architecture**

Set Accelerator → **GPU T4 x1** before running.

| Phase | What | Time |
|-------|------|------|
| Setup | Clone, install, build model | ~5 min |
| Train | TinyStories 50M tokens, 5000 steps | ~3 hrs |
| Eval  | Generation + efficiency + ablation | ~15 min |

In [ ]:
# Cell 1: Clone repo + load modules
import os, sys, importlib.util
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
os.environ['PYTORCH_ALLOC_CONF']   = 'expandable_segments:True'

# Fresh clone every run
os.system('rm -rf /kaggle/working/MobileLLM')
os.system('git clone https://github.com/opszx/MobileLLM.git /kaggle/working/MobileLLM')
REPO = '/kaggle/working/MobileLLM'

# Load modules from cloned repo
def load_mod(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m    = importlib.util.module_from_spec(spec)
    sys.modules[name] = m
    spec.loader.exec_module(m)
    return m

load_mod('bitlinear',   f'{REPO}/bitlinear.py')
load_mod('mamba_block', f'{REPO}/mamba_block.py')
load_mod('attention',   f'{REPO}/attention.py')
load_mod('moe',         f'{REPO}/moe.py')
cfg_m   = load_mod('config', f'{REPO}/config.py')
mdl_m   = load_mod('model',  f'{REPO}/model.py')

PhantomLMConfig = cfg_m.PhantomLMConfig
PhantomLM       = mdl_m.PhantomLM
print('Repo cloned and modules loaded OK')

In [ ]:
# Cell 2: Install deps
os.system('pip install einops datasets transformers sentencepiece -q')
print('Done')

In [ ]:
# Cell 3: GPU check
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_mem/1024**3:.1f}GB')
else:
    print('WARNING: No GPU — enable T4 in Settings')

In [ ]:
# Cell 4: Tokenizer
from transformers import AutoTokenizer
tokenizer           = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE          = tokenizer.vocab_size
print(f'Tokenizer: GPT-2  vocab={VOCAB_SIZE}')

In [ ]:
# Cell 5: Build model — Phase 1 (full precision)
# Disable quantization for architecture validation.
# Re-enable in Phase 2 to benchmark quantized efficiency.
import bitlinear
bitlinear.QUANTIZE_ENABLED = False

config = PhantomLMConfig.phantom_medium()  # ~50M params
config.vocab_size        = VOCAB_SIZE
config.max_seq_len       = 256
config.batch_size        = 8
config.d_state           = 16
config.moe_aux_loss_coef = 0.001
config.max_steps         = 5000
config.warmup_steps      = 300
config.learning_rate     = 3e-4
config.grad_clip         = 1.0

model = PhantomLM(config).to(device).to(torch.bfloat16)
params = sum(p.numel() for p in model.parameters())
used   = torch.cuda.memory_allocated() / 1024**3
print(f'\nParameters: {params:,}  VRAM: {used:.2f}GB')

# Sanity check
x = torch.randint(0, config.vocab_size, (2, 32), device=device)
loss, _, lm, aux = model(x, targets=x, use_checkpoint=True)
print(f'Sanity check OK  loss={lm.item():.4f}  aux={aux.item():.4f}')
loss.backward()
dt_has_grad = any(p.grad is not None and p.grad.abs().sum() > 0
                  for n, p in model.named_parameters() if 'dt_proj' in n)
print(f'Mamba dt_proj has gradients: {dt_has_grad}')
assert dt_has_grad, 'Mamba backward is still broken!'
del x; torch.cuda.empty_cache()
model.zero_grad(set_to_none=True)

In [ ]:
# Cell 6: TinyStories dataset — 50M tokens
from datasets import load_dataset
from torch.utils.data import IterableDataset

print('Loading TinyStories (streaming)...')
stories = load_dataset('roneneldan/TinyStories', split='train', streaming=True)

class TinyStoriesDataset(IterableDataset):
    def __init__(self, ds, tok, seq_len, max_tok=50_000_000):
        self.ds, self.tok, self.seq_len = ds, tok, seq_len
        self.max_tok = max_tok
        self.eos = tok.eos_token_id

    def __iter__(self):
        buf, total = [], 0
        for item in self.ds:
            if total >= self.max_tok:
                return
            text = item.get('text', '').strip()
            if not text:
                continue
            toks = self.tok.encode(text) + [self.eos]
            buf += toks
            total += len(toks)
            while len(buf) >= self.seq_len + 1:
                chunk = buf[:self.seq_len + 1]
                buf = buf[self.seq_len:]
                yield (torch.tensor(chunk[:-1], dtype=torch.long),
                       torch.tensor(chunk[1:],  dtype=torch.long))

train_dataset = TinyStoriesDataset(stories, tokenizer, config.max_seq_len, 50_000_000)
print(f'TinyStories ready — 50M tokens, seq_len={config.max_seq_len}')

In [ ]:
# Cell 7: Trainer
import math, time
from torch.optim import AdamW
from torch.utils.data import DataLoader

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

class PhantomTrainer:
    def __init__(self, model, config, train_dataset,
                 checkpoint_dir=CHECKPOINT_DIR,
                 grad_accum=4, log_every=50, save_every=1000):
        self.model       = model
        self.config      = config
        self.ckpt_dir    = checkpoint_dir
        self.grad_acc    = grad_accum
        self.log_every   = log_every
        self.save_every  = save_every
        self.step        = 0
        self.best_loss   = float('inf')
        self.loader      = DataLoader(
            train_dataset, batch_size=config.batch_size,
            num_workers=2, pin_memory=True
        )
        self.optimizer = AdamW(
            model.parameters(), lr=config.learning_rate,
            weight_decay=0.1, betas=(0.9, 0.95)
        )

    def get_lr(self, step):
        max_lr = self.config.learning_rate
        min_lr = max_lr * 0.1
        w, T   = self.config.warmup_steps, self.config.max_steps
        if step < w:
            return max_lr * step / max(w, 1)
        if step >= T:
            return min_lr
        return min_lr + 0.5 * (max_lr - min_lr) * (
            1 + math.cos(math.pi * (step - w) / (T - w)))

    def save(self, tag):
        path = os.path.join(self.ckpt_dir, f'phantomlm_{tag}.pt')
        torch.save({
            'step': self.step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'config': self.config,
            'best_loss': self.best_loss
        }, path)
        print(f'  Saved: {path}')
        # Delete old rolling checkpoint
        if tag.startswith('step_'):
            old = os.path.join(self.ckpt_dir,
                f'phantomlm_step_{self.step - self.save_every}.pt')
            if os.path.exists(old):
                os.remove(old)

    def resume(self, path):
        if not os.path.exists(path):
            print('No checkpoint found — starting fresh')
            return
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        self.model.load_state_dict(ckpt['model_state_dict'])
        self.optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        self.step = ckpt['step']+
        self.best_loss = ckpt.get('best_loss', float('inf'))
        print(f'Resumed from step {self.step}')

    def train(self):
        """Train until max_steps. Loops over dataset if needed."""
        self.model.train()
        self.optimizer.zero_grad()
        t0, total_tokens, batch_idx = time.time(), 0, 0

        for epoch in range(999):
            for x, y in self.loader:
                if self.step >= self.config.max_steps:
                    print(f'\nTraining complete at step {self.step}.')
                    self.save('final')
                    return

                x, y = x.to(device), y.to(device)
                for pg in self.optimizer.param_groups:
                    pg['lr'] = self.get_lr(self.step)

                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    loss, _, lm_loss, aux_loss = self.model(
                        x, targets=y, use_checkpoint=True)

                (loss / self.grad_acc).backward()
                total_tokens += x.numel()
                batch_idx += 1

                if batch_idx % self.grad_acc == 0:
                    gn = torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), self.config.grad_clip)
                    self.optimizer.step()
                    self.optimizer.zero_grad()
                    self.step += 1

                    # Clear cache periodically (not every step)
                    if self.step % 100 == 0:
                        torch.cuda.empty_cache()

                    if self.step % self.log_every == 0:
                        elapsed = time.time() - t0
                        tok_s = total_tokens / elapsed
                        vram  = torch.cuda.memory_allocated() / 1024**3
                        print(
                            f'Step {self.step:5d} | '
                            f'loss:{lm_loss.item():.4f} | '
                            f'aux:{aux_loss.item():.4f} | '
                            f'lr:{self.get_lr(self.step):.2e} | '
                            f'norm:{gn:.3f} | '
                            f'tok/s:{tok_s:,.0f} | '
                            f'VRAM:{vram:.1f}GB')
                        t0, total_tokens = time.time(), 0

                    if self.step % self.save_every == 0:
                        self.save(f'step_{self.step}')
                        if lm_loss.item() < self.best_loss:
                            self.best_loss = lm_loss.item()
                            self.save('best')

            print(f'  [Epoch done at step {self.step}, restarting dataset]')

print('Trainer ready')

In [ ]:
# Cell 8: Train — TinyStories pretraining
# Expected: ~3 hours on T4, target loss: ~3.0-4.0
trainer = PhantomTrainer(
    model, config, train_dataset, CHECKPOINT_DIR,
    grad_accum=4, log_every=50, save_every=1000
)
trainer.resume(f'{CHECKPOINT_DIR}/phantomlm_best.pt')

print(f'Training: step {trainer.step} → {config.max_steps}')
print(f'Estimated time: ~3 hours')
print(f'Quantization: {"ON" if bitlinear.QUANTIZE_ENABLED else "OFF (full precision)"}')
print('-' * 60)

trainer.train()

In [ ]:
# Cell 9: Generation test
model.eval()

def generate(prompt, max_new=100, temp=0.8):
    toks = tokenizer.encode(prompt)
    ids  = torch.tensor([toks], device=device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new,
            temperature=temp, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, len(toks):].tolist(), skip_special_tokens=True)

print('=== Generation Test ===')
for prompt in [
    'Once upon a time there was a',
    'The little dog ran to the',
    'One day a girl named Lily',
]:
    print(f'\nPrompt  : {prompt}')
    print(f'Response: {generate(prompt)}')
    print('-' * 60)

model.train()

In [ ]:
# Cell 10: Efficiency benchmark + ablation
# This fills the paper comparison table — no extra training needed.
import copy, json
model.eval()

print('\n' + '=' * 70)
print('EFFICIENCY BENCHMARK')
print('=' * 70)
print(f'{"Context":>10} {"Tok/s":>10} {"Latency":>12} {"VRAM MB":>10}')
print('-' * 46)

for seq_len in [32, 64, 128, 256]:
    if seq_len > config.max_seq_len:
        continue
    dummy = torch.randint(0, config.vocab_size, (1, seq_len), device=device)
    # Warmup
    with torch.no_grad():
        _ = model(dummy, return_loss=False)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    # Benchmark
    import time as _t
    t0 = _t.perf_counter()
    with torch.no_grad():
        for _ in range(50):
            _ = model(dummy, return_loss=False)
    torch.cuda.synchronize()
    elapsed = _t.perf_counter() - t0
    vram = torch.cuda.max_memory_allocated() / 1024**2
    print(f'{seq_len:>10} {seq_len*50/elapsed:>10.0f} '
          f'{elapsed/50*1000:>11.1f}ms {vram:>10.1f}')

params = sum(p.numel() for p in model.parameters())
print(f'\nModel: {params:,} params')
print(f'FP32 size:  {params * 4 / 1024**2:.1f} MB')
print(f'BF16 size:  {params * 2 / 1024**2:.1f} MB')
print(f'Quantized:  {params * 0.25 / 1024**2:.1f} MB (estimated 1.58-bit avg)')

In [ ]:
# Cell 11: Ablation study — architecture variants
# Creates modified versions of your model to compare design choices.
# NO extra training needed — uses untrained variants for efficiency comparison.
from attention import GroupedQueryAttention

def measure(name, m):
    m = m.to(device).to(torch.bfloat16).eval()
    p = sum(x.numel() for x in m.parameters())
    dummy = torch.randint(0, config.vocab_size, (1, 128), device=device)
    # Warmup
    with torch.no_grad():
        _ = m(dummy, return_loss=False)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = _t.perf_counter()
    with torch.no_grad():
        for _ in range(50):
            _ = m(dummy, return_loss=False)
    torch.cuda.synchronize()
    elapsed = _t.perf_counter() - t0
    tok_s = 128 * 50 / elapsed
    vram  = torch.cuda.max_memory_allocated() / 1024**2
    del m; torch.cuda.empty_cache()
    return {'params': p, 'tok_s': round(tok_s), 'vram_mb': round(vram, 1)}

results = {}

# 1. PhantomLM (your trained model)
results['PhantomLM (full)'] = measure('PhantomLM', model)

# 2. Pure Transformer — all Mamba replaced with Attention
class PureTransformer(PhantomLM):
    def __init__(self, cfg):
        super().__init__(cfg)
        for layer in self.layers:
            if layer.layer_type == 'mamba':
                layer.core = GroupedQueryAttention(cfg)
                layer.layer_type = 'attention'
results['Pure Transformer'] = measure('Pure Transformer',
    PureTransformer(config))

# 3. No MoE — standard FFN everywhere
cfg3 = copy.deepcopy(config)
cfg3.moe_every_n_layers = 9999
results['No MoE'] = measure('No MoE', PhantomLM(cfg3))

# 4. Uniform attention (no zones)
cfg4 = copy.deepcopy(config)
cfg4.mamba_zone_end = 0
cfg4.mixed_zone_end = config.n_layers
results['Uniform placement'] = measure('Uniform', PhantomLM(cfg4))

# Print comparison table
print('\n' + '=' * 65)
print('ARCHITECTURE COMPARISON')
print('=' * 65)
print(f'{"Variant":<25} {"Params":>12} {"Tok/s":>8} {"VRAM MB":>10}')
print('-' * 65)
for name, m in results.items():
    print(f'{name:<25} {m["params"]:>12,} {m["tok_s"]:>8,} {m["vram_mb"]:>10.1f}')
print('=' * 65)

In [ ]:
# Cell 12: Save everything
import json

final_results = {
    'model': 'PhantomLM-medium',
    'params': sum(p.numel() for p in model.parameters()),
    'training_steps': trainer.step,
    'best_loss': trainer.best_loss,
    'quantization': bitlinear.QUANTIZE_ENABLED,
    'ablation': results,
}

with open('/kaggle/working/paper_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print('Results saved to paper_results.json')
print(json.dumps(final_results, indent=2))